In [ ]:
import glob
import pandas
columns_to_discard = [
    'collision_index',
    'collision_ref_no',
    'location_easting_osgr',
    'location_northing_osgr',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'lsoa_of_accident_location',
    'AREFNO'
]

data_frames = []
file_paths = glob.glob('*.csv')

if not file_paths:
    if os.path.exists('merged_collision_data.csv'):
        file_paths = ['merged_collision_data.csv']
    else:
        raise FileNotFoundError("No CSV files found in current directory and 'merged_collision_data.csv' does not exist.")

for file_path in file_paths:
    try:
        df = pd.read_csv(file_path, low_memory=False)
        cols_present = [col for col in columns_to_discard if col in df.columns]
        if cols_present:
            df = df.drop(columns=cols_present, axis=1)
        data_frames.append(df)
        print(f"Loaded {file_path} with shape {df.shape}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

merged_df = pd.concat(data_frames, ignore_index=True)
merged_df.to_csv('merged_collision_data.csv', index=False)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras

# ----------------------------
# Columns to drop (from your earlier snippet)
# ----------------------------
columns_to_discard = [
    'collision_index',
    'collision_ref_no',
    'location_easting_osgr',
    'location_northing_osgr',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'lsoa_of_accident_location',
    'AREFNO'
]

# ----------------------------
# 1) Read merged CSV
# ----------------------------
merged_path = 'merged_collision_data.csv'
if not os.path.exists(merged_path):
    raise FileNotFoundError(f"{merged_path} not found in current directory.")

merged_df = pd.read_csv(merged_path, low_memory=False)
print(f"Loaded merged file -> shape: {merged_df.shape}")
print(merged_df.head(5))

# Drop specified columns if present
cols_present = [c for c in columns_to_discard if c in merged_df.columns]
if cols_present:
    merged_df = merged_df.drop(columns=cols_present)
    print(f"Dropped columns_to_discard: {cols_present}")

# ----------------------------
# 2) Normalize target to 'Severity'
# ----------------------------
if 'Severity' not in merged_df.columns and 'collision_severity' in merged_df.columns:
    merged_df['Severity'] = merged_df['collision_severity']

if 'collision_severity' in merged_df.columns:
    merged_df = merged_df.drop(columns=['collision_severity'])

merged_df = merged_df.dropna(subset=['Severity'])
# keep original if numeric; otherwise uppercase string
merged_df['Severity'] = merged_df['Severity'].apply(lambda v: str(v).upper().strip())

print("\nTarget distribution (raw):")
print(merged_df['Severity'].value_counts())

# ----------------------------
# 3) Optional: extract hour/month if date/time exist
# ----------------------------
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], format='%d/%m/%Y', errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')
    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month
    merged_df = merged_df.drop(columns=['date', 'time'])
    print("Extracted 'hour_of_day' and 'month'")

# ----------------------------
# 4) Build X_df and handle numeric/categorical per your "no imputation" rule:
#    - Drop numeric columns that contain any NaN (so no imputing numeric values)
#    - Keep categorical columns (NaNs will be mapped to code 0)
# ----------------------------
X_df = merged_df.drop(columns=['Severity']).copy()

# coerce clearly numeric columns to numeric (strings that are numeric -> numeric, else NaN)
for col in X_df.columns:
    # only try coercion for object columns — don't change existing numeric dtypes
    if X_df[col].dtype == 'object' or pd.api.types.is_string_dtype(X_df[col]):
        # quick check: if many values look numeric, coerce
        non_null = X_df[col].dropna().astype(str)
        if len(non_null) > 0:
            parsed = non_null.str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
            try:
                frac_numeric = parsed.notna().sum() / len(non_null)
            except Exception:
                frac_numeric = 0.0
            if frac_numeric > 0.9:
                X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

# identify numeric and categorical columns
numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

# drop numeric columns that contain any NaN (no numeric imputation allowed)
numeric_cols_with_nan = [c for c in numeric_cols if X_df[c].isnull().any()]
if numeric_cols_with_nan:
    print(f"Dropping numeric columns that contain NaN (no imputations): {numeric_cols_with_nan}")
    X_df = X_df.drop(columns=numeric_cols_with_nan)
    # recompute numeric/categorical lists
    numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"\nRemaining numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Remaining categorical columns ({len(categorical_cols)}): {categorical_cols}")

# ensure we have at least some features
if len(numeric_cols) + len(categorical_cols) == 0:
    raise RuntimeError("No features remain after dropping numeric columns containing NaN and discarding others. "
                       "Allow imputations or provide a dataset with usable features.")

# keep full aligned target series
y_series = merged_df['Severity'].reset_index(drop=True)

# ----------------------------
# 5) Label-encode target (for StratifiedKFold)
# ----------------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y_series.values)
num_classes = len(np.unique(y_encoded))
if num_classes < 2:
    raise RuntimeError("Less than two target classes found after cleaning. Cannot train classifier.")
print(f"\nTarget classes: {list(le.classes_)} (num_classes={num_classes})")

# If some class has fewer than 5 samples, reduce n_splits accordingly
counts = np.bincount(y_encoded)
min_count = counts.min()
n_splits = 5
if min_count < n_splits:
    n_splits = int(min_count)
    if n_splits < 2:
        raise RuntimeError("Not enough samples per class to perform CV. Need at least 2 samples in smallest class.")
    print(f"Reduced StratifiedKFold n_splits to {n_splits} because smallest class has {min_count} samples.")

# ----------------------------
# 6) Helpers: per-fold categorical mapping (no imputation)
# ----------------------------
def fit_cat_mappings(df, cat_cols):
    mappings = {}
    for col in cat_cols:
        uniques = pd.Series(df[col].astype(str).fillna("")).unique().tolist()
        mappings[col] = {val: idx + 1 for idx, val in enumerate(uniques)}  # codes 1..K
    return mappings

def apply_cat_mappings(df, mappings, cat_cols):
    if not cat_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    out = np.zeros((len(df), len(cat_cols)), dtype=np.float32)
    for j, col in enumerate(cat_cols):
        series = df[col].astype(str).fillna("")
        mapdict = mappings[col]
        mapped = series.map(lambda x: mapdict.get(x, 0))
        out[:, j] = mapped.astype(np.float32)
    return out

# ----------------------------
# 7) Model factory
# ----------------------------
def create_model(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(128, activation='relu')(inputs)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------
# 8) Stratified CV
# ----------------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True)

all_acc = []
all_loss = []
last_history = None
last_model = None
last_X_val = None
last_y_val = None

# For consistent indexing use the current X_df built from merged_df
X_df = X_df.reset_index(drop=True)
y_for_cv = y_encoded  # labelencoded aligned with merged_df index

for fold, (train_idx, val_idx) in enumerate(skf.split(X_df, y_for_cv), start=1):
    print(f"\n--- Fold {fold} ---")
    X_train_df = X_df.iloc[train_idx].reset_index(drop=True)
    X_val_df = X_df.iloc[val_idx].reset_index(drop=True)
    y_train = y_for_cv[train_idx]
    y_val = y_for_cv[val_idx]

    # fit categorical mappings on training fold only
    cat_mappings = fit_cat_mappings(X_train_df, categorical_cols)

    # apply categorical mappings (unknown -> 0)
    X_train_cat = apply_cat_mappings(X_train_df, cat_mappings, categorical_cols)
    X_val_cat = apply_cat_mappings(X_val_df, cat_mappings, categorical_cols)

    # numeric scaling (train statistics; no imputation)
    if numeric_cols:
        X_train_num = X_train_df[numeric_cols].astype(float).to_numpy()
        X_val_num = X_val_df[numeric_cols].astype(float).to_numpy()
        means = np.nanmean(X_train_num, axis=0)
        stds = np.nanstd(X_train_num, axis=0)
        stds[stds == 0] = 1.0
        X_train_num = ((X_train_num - means) / stds).astype(np.float32)
        X_val_num = ((X_val_num - means) / stds).astype(np.float32)
    else:
        X_train_num = np.zeros((len(X_train_df), 0), dtype=np.float32)
        X_val_num = np.zeros((len(X_val_df), 0), dtype=np.float32)

    # combine features: numeric first, then categorical codes
    X_train = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
    X_val = np.hstack([X_val_num, X_val_cat]).astype(np.float32)

    print(f"Fold shapes -> train: {X_train.shape}, val: {X_val.shape}; classes in train: {np.unique(y_train)}")

    # create and train model
    model = create_model(X_train.shape[1], num_classes)

    cw = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    cw_dict = {i: w for i, w in enumerate(cw)}

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        class_weight=cw_dict,
        verbose=1
    )

    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold} -> val_loss: {loss:.4f}, val_acc: {acc:.4f}")
    all_acc.append(acc)
    all_loss.append(loss)

    last_history = history
    last_model = model
    last_X_val = X_val
    last_y_val = y_val

# ----------------------------
# 9) Summary + final evaluation
# ----------------------------
print(f"\nAverage Validation Accuracy: {np.mean(all_acc):.4f} (+/- {np.std(all_acc):.4f})")
print(f"Average Validation Loss: {np.mean(all_loss):.4f} (+/- {np.std(all_loss):.4f})")

if last_history is not None:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(last_history.history.get('accuracy', []), label='train_acc')
    plt.plot(last_history.history.get('val_accuracy', []), label='val_acc')
    plt.legend(); plt.title('Last fold acc')
    plt.subplot(1,2,2)
    plt.plot(last_history.history.get('loss', []), label='train_loss')
    plt.plot(last_history.history.get('val_loss', []), label='val_loss')
    plt.legend(); plt.title('Last fold loss')
    plt.tight_layout(); plt.show()

if last_model is not None and last_X_val is not None:
    y_pred_prob = last_model.predict(last_X_val)
    y_pred = np.argmax(y_pred_prob, axis=1)

    unique_labels_in_fold = np.unique(last_y_val)
    target_names_in_fold = le.inverse_transform(unique_labels_in_fold)

    print("\nClassification report (last fold):")
    print(classification_report(last_y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

    cm = confusion_matrix(last_y_val, y_pred)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names_in_fold, yticklabels=target_names_in_fold)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (last fold)'); plt.show()
